In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4


In [ ]:
import zipfile
import os

zip_path = "/content/ECHO-AI-pipeline.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted successfully!")
print("Contents:", os.listdir("/content/ECHO-AI"))

ZIP extracted successfully!
Contents: ['requirements.txt', 'data', 'app', 'models', 'src', '.gitignore']


In [ ]:
import os

print("Files in /content:")
print(os.listdir("/content/"))

Files in /content:
['.config', 'ECHO-AI', 'ECHO-AI-pipeline.zip', 'sample_data']


In [ ]:
import os

project_path = "/content/ECHO-AI"

for root, dirs, files in os.walk(project_path):
    dirs[:] = [d for d in dirs if d not in [".venv", "__pycache__"]]
    level = root.replace(project_path, "").count(os.sep)

    if level > 3:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(root) + "/")
    for file in files:
        print("  " * (level + 1) + file)

ECHO-AI/
  requirements.txt
  .gitignore
  data/
    README.md
    .gitkeep
  app/
    streamlit_app.py
  models/
    README.md
  src/
    __init__.py
    train.py
    data_preprocessing.py
    gradcam.py
    evaluate.py
    model.py


In [ ]:
with open("/content/ECHO-AI/src/train.py", "r") as f:
    print(f.read())

"""Train DenseNet121 for multi-label chest X-ray classification.

Example:
    python -m src.train --data-root /content/nih --epochs 5 --batch-size 64
"""
import argparse
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from .data_preprocessing import LABELS, CXRDataset, build_transforms, load_metadata, make_splits
from .evaluate import evaluate_split, macro_auc, predict, tune_thresholds
from .model import build_model, save_checkpoint


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", required=True)
    ap.add_argument("--out", default="models/echo_densenet121.pt")
    ap.add_argument("--out-dir", default="outputs")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--bat

In [ ]:
with open("/content/ECHO-AI/src/data_preprocessing.py", "r") as f:
    print(f.read())

"""Data loading, label encoding, patient-level splits and transforms for NIH ChestX-ray14."""
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import Dataset
from torchvision import transforms as T

# Names exactly as they appear in Data_Entry_2017.csv ("Finding Labels").
LABELS = ["Pneumonia", "Cardiomegaly", "Effusion", "Edema",
          "Pneumothorax", "Atelectasis", "Consolidation"]
DISPLAY_NAMES = {"Effusion": "Pleural Effusion"}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_EXTS = {".png", ".jpg", ".jpeg"}


def display_name(label):
    return DISPLAY_NAMES.get(label, label)


def build_transforms(img_size=224, train=False):
    """Eval: resize + normalize. Train: mild augmentation that keeps anatomy plausible
    (no horizontal flip, because the heart sits on the left side of the image)."""
    norm = T.Normal

In [ ]:
!pip -q install -U kaggle

import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

!kaggle --version

Kaggle CLI 2.2.5


In [ ]:
!kaggle datasets list -s "NIH Chest X-rays"

ref                                                     title                                            size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------  ----------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
nih-chest-xrays/data                                    NIH Chest X-rays                          45096150231  2018-02-21 20:52:23.880000         545570       1680  0.7352941        
nickuzmenkov/nih-chest-xrays-tfrecords                  NIH Chest X-rays TFRecords                11984003628  2021-03-09 04:49:23.033000           2738        131  0.9411765        
huthayfahodeb/nih-chest-x-rays-bbox-version             NIH Chest X-rays Bbox version               758359590  2024-06-25 17:59:24.140000           1207         30  1                
rahulogoel/clahe-enhancement-on-chestx-ray14            NIH CLAHE Enhanced Chest X-ra

In [ ]:
!kaggle datasets files -d nih-chest-xrays/data

Next Page Token = CfDJ8FwkkMkekkBCtzoFDl4Vj7jnhkv0EUeSdJ8g266oNVBlN5KuwLhqpxGuffadsqRsnDg0sdMedVxRdz1yKPmTe08bA6ctvJObyaeQn9ke898ZhIvDIHhVJtIjYahPHfl4syscQFSGyRZAcRC957vHsh3F-Czj
name                                   size  creationDate                
----------------------------------  -------  --------------------------  
ARXIV_V5_CHESTXRAY.pdf              8963138  2019-09-24 23:56:19.481000  
BBox_List_2017.csv                    92416  2019-09-24 23:56:19.362000  
Data_Entry_2017.csv                 7861152  2019-09-24 23:56:19.543000  
FAQ_CHESTXRAY.pdf                     72223  2019-09-24 23:56:19.721000  
LOG_CHESTXRAY.pdf                      3919  2019-09-24 23:56:19.430000  
README_CHESTXRAY.pdf                 847223  2019-09-24 23:56:19.407000  
images_001/images/00000001_000.png   457171  2018-02-21 23:57:46.601000  
images_001/images/00000001_001.png   444655  2018-02-21 23:56:51.890000  
images_001/images/00000001_002.png   351772  2018-02-21 23:58:27.109000  
images_

In [ ]:
!kaggle datasets files -d nih-chest-xrays/data

Next Page Token = CfDJ8FwkkMkekkBCtzoFDl4Vj7hl6pfBaWtjpJCrt7_dbZN8AUoTcuHEbzDeAtYRrOckSrtuIXTD_M91XeCOkb5z9IhN9YDx36F4zVjKOXM03m9Y-1JUw3VB-ASH9Bgh853u0-w6frnJMx14EiT6jPbPvZGMwYy8
name                                   size  creationDate                
----------------------------------  -------  --------------------------  
ARXIV_V5_CHESTXRAY.pdf              8963138  2019-09-24 23:56:19.481000  
BBox_List_2017.csv                    92416  2019-09-24 23:56:19.362000  
Data_Entry_2017.csv                 7861152  2019-09-24 23:56:19.543000  
FAQ_CHESTXRAY.pdf                     72223  2019-09-24 23:56:19.721000  
LOG_CHESTXRAY.pdf                      3919  2019-09-24 23:56:19.430000  
README_CHESTXRAY.pdf                 847223  2019-09-24 23:56:19.407000  
images_001/images/00000001_000.png   457171  2018-02-21 23:57:46.601000  
images_001/images/00000001_001.png   444655  2018-02-21 23:56:51.890000  
images_001/images/00000001_002.png   351772  2018-02-21 23:58:27.109000  
images_

In [ ]:
!mkdir -p /content/nih

!kaggle datasets download \
  -d nih-chest-xrays/data \
  -f Data_Entry_2017.csv \
  -p /content/nih

Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/data
License(s): CC0-1.0
100% 924k/924k [00:00<00:00, 151MB/s]



In [ ]:
!unzip -o /content/nih/Data_Entry_2017.csv.zip -d /content/nih

Archive:  /content/nih/Data_Entry_2017.csv.zip
  inflating: /content/nih/Data_Entry_2017.csv  


In [ ]:
import pandas as pd

df = pd.read_csv("/content/nih/Data_Entry_2017.csv")

print("Total records:", len(df))
print("Columns:", df.columns.tolist())
print(df[["Image Index", "Finding Labels", "Patient ID"]].head())

Total records: 112120
Columns: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']
        Image Index          Finding Labels  Patient ID
0  00000001_000.png            Cardiomegaly           1
1  00000001_001.png  Cardiomegaly|Emphysema           1
2  00000001_002.png   Cardiomegaly|Effusion           1
3  00000002_000.png              No Finding           2
4  00000003_000.png                  Hernia           3


In [ ]:
from collections import Counter

label_counts = Counter()

for labels in df["Finding Labels"].dropna():
    for label in labels.split("|"):
        label_counts[label.strip()] += 1

print("Unique findings:", len(label_counts))
print("\nMost common findings:")

for label, count in label_counts.most_common(20):
    print(f"{label}: {count}")

Unique findings: 15

Most common findings:
No Finding: 60361
Infiltration: 19894
Effusion: 13317
Atelectasis: 11559
Nodule: 6331
Mass: 5782
Pneumothorax: 5302
Consolidation: 4667
Pleural_Thickening: 3385
Cardiomegaly: 2776
Emphysema: 2516
Edema: 2303
Fibrosis: 1686
Pneumonia: 1431
Hernia: 227


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   71G  38% /


In [ ]:
!kaggle datasets files -d nih-chest-xrays/data | grep "images_001"

images_001/images/00000001_000.png   457171  2018-02-21 23:57:46.601000  
images_001/images/00000001_001.png   444655  2018-02-21 23:56:51.890000  
images_001/images/00000001_002.png   351772  2018-02-21 23:58:27.109000  
images_001/images/00000002_000.png   402332  2018-02-21 23:56:53.013000  
images_001/images/00000003_000.png   450000  2018-02-21 23:57:38.618000  
images_001/images/00000003_001.png   383773  2018-02-21 23:58:14.255000  
images_001/images/00000003_002.png   343273  2018-02-21 23:57:05.214000  
images_001/images/00000003_003.png   456605  2018-02-21 23:57:01.077000  
images_001/images/00000003_004.png   371185  2018-02-21 23:58:04.438000  
images_001/images/00000003_005.png   437334  2018-02-21 23:57:00.090000  
images_001/images/00000003_006.png   413626  2018-02-21 23:58:24.598000  
images_001/images/00000003_007.png   461458  2018-02-21 23:57:35.285000  
images_001/images/00000004_000.png   601599  2018-02-21 23:58:00.195000  
images_001/images/00000005_000.png   3

In [ ]:
!kaggle datasets files -d nih-chest-xrays/data > /content/kaggle_files.txt

with open("/content/kaggle_files.txt", "r") as f:
    lines = f.readlines()

archives = [
    line.strip() for line in lines
    if any(ext in line.lower() for ext in [".zip", ".tar", ".7z"])
]

print("Archives found:", len(archives))
for item in archives[:30]:
    print(item)

Archives found: 0


In [ ]:
!pip -q install -U kagglehub

import kagglehub

print("KaggleHub version:", kagglehub.__version__)

KaggleHub version: 1.0.3


In [ ]:
import kagglehub

try:
    path = kagglehub.dataset_download(
        "nih-chest-xrays/data",
        output_dir="/content/nih_kagglehub_test",
        force_download=False
    )
    print("Dataset path:", path)
except Exception as e:
    print("Download check failed:", type(e).__name__, str(e)[:500])

100%|██████████| 42.0G/42.0G [08:42<00:00, 86.4MB/s]

Extracting files...


Download check failed: OSError [Errno 28] No space left on device


In [ ]:
import os

path = "/content/nih_kagglehub_test"

print("Folder exists:", os.path.exists(path))

if os.path.exists(path):
    for root, dirs, files in os.walk(path):
        print("\nFolder:", root)
        print("Files:", files[:10])
        print("Subfolders:", dirs[:10])
        if root.count(os.sep) - path.count(os.sep) >= 2:
            dirs[:] = []


Folder exists: True

Folder: /content/nih_kagglehub_test
Files: ['README_CHESTXRAY.pdf', '3.archive', 'Data_Entry_2017.csv', 'LOG_CHESTXRAY.pdf', 'ARXIV_V5_CHESTXRAY.pdf', 'FAQ_CHESTXRAY.pdf', 'BBox_List_2017.csv']
Subfolders: ['images_006', 'images_007', 'images_001', 'images_005', 'images_004', 'images_003', 'images_002', 'images_009', 'images_008']

Folder: /content/nih_kagglehub_test/images_006
Files: []
Subfolders: ['images']

Folder: /content/nih_kagglehub_test/images_006/images
Files: ['00013006_000.png', '00012376_010.png', '00013708_003.png', '00012610_005.png', '00012158_021.png', '00013591_000.png', '00011598_006.png', '00012697_001.png', '00012072_013.png', '00012706_000.png']
Subfolders: []

Folder: /content/nih_kagglehub_test/images_007
Files: []
Subfolders: ['images']

Folder: /content/nih_kagglehub_test/images_007/images
Files: ['00014986_001.png', '00014331_001.png', '00015706_000.png', '00014798_006.png', '00015530_066.png', '00014380_004.png', '00015676_000.png', '00

In [ ]:
import os
import pandas as pd

DATA_ROOT = "/content/nih_kagglehub_test"

df = pd.read_csv(os.path.join(DATA_ROOT, "Data_Entry_2017.csv"))

image_count = 0
for root, dirs, files in os.walk(DATA_ROOT):
    image_count += sum(
        1 for f in files
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    )

print("Metadata records:", len(df))
print("Downloaded images:", image_count)
print("Dataset folder:", DATA_ROOT)
print("CSV exists:", os.path.exists(
    os.path.join(DATA_ROOT, "Data_Entry_2017.csv")
))

Metadata records: 112120
Downloaded images: 75821
Dataset folder: /content/nih_kagglehub_test
CSV exists: True


In [ ]:
import os
import pandas as pd

DATA_ROOT = "/content/nih_kagglehub_test"

df = pd.read_csv(os.path.join(DATA_ROOT, "Data_Entry_2017.csv"))

available_images = set()

for root, _, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            available_images.add(f)

df["Image Available"] = df["Image Index"].isin(available_images)

print("Total metadata records:", len(df))
print("Records with images:", df["Image Available"].sum())
print("Records missing images:", (~df["Image Available"]).sum())
print("\nAvailable records by finding:")
print(
    df[df["Image Available"]]
    ["Finding Labels"]
    .value_counts()
    .head(10)
)

Total metadata records: 112120
Records with images: 75821
Records missing images: 36299

Available records by finding:
Finding Labels
No Finding                  41893
Infiltration                 5431
Atelectasis                  2902
Effusion                     2688
Nodule                       1819
Mass                         1540
Pneumothorax                 1390
Effusion|Infiltration         969
Consolidation                 923
Atelectasis|Infiltration      877
Name: count, dtype: int64


In [ ]:
%cd /content/ECHO-AI

!grep -n -E "def load_metadata|def make_splits|class ChestXrayDataset|LABELS|train_test_split|GroupShuffleSplit" src/data_preprocessing.py

OSError: [Errno 28] No space left on device: '/root/.ipython/profile_default/db/dhist'

In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  113G     0 100% /


In [ ]:
!du -xhd1 /content 2>/dev/null | sort -h

72K	/content/ECHO-AI
148K	/content/.config
8.5M	/content/nih
55M	/content/sample_data
71G	/content
71G	/content/nih_kagglehub_test


In [ ]:
!du -xhd1 /root/.cache 2>/dev/null | sort -h

4.0K	/root/.cache/uv
8.0K	/root/.cache/antigravity
32K	/root/.cache/matplotlib
828K	/root/.cache/pip
56M	/root/.cache/node-gyp
57M	/root/.cache


In [ ]:
!find /content/nih_kagglehub_test -maxdepth 2 -type f -printf '%s %p\n' 2>/dev/null | sort -nr | head -20

45096150231 /content/nih_kagglehub_test/3.archive
8963138 /content/nih_kagglehub_test/ARXIV_V5_CHESTXRAY.pdf
7861152 /content/nih_kagglehub_test/Data_Entry_2017.csv
847223 /content/nih_kagglehub_test/README_CHESTXRAY.pdf
92416 /content/nih_kagglehub_test/BBox_List_2017.csv
72223 /content/nih_kagglehub_test/FAQ_CHESTXRAY.pdf
3919 /content/nih_kagglehub_test/LOG_CHESTXRAY.pdf


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  113G     0 100% /


In [ ]:
import os
import shutil

paths = [
    "/content/ECHO-AI/ECHO-AI/models/echo_densenet121.pt",
    "/content/ECHO-AI/ECHO-AI/models/history.json",
    "/content/nih_sample/Data_Entry_2017.csv",
    "/content/nih_sample/images",
    "/content/nih_kagglehub_test/3.archive",
]

print("=== SAVED FILES ===")
for path in paths:
    if os.path.isfile(path):
        print(f"FOUND: {path} ({os.path.getsize(path) / (1024**2):.2f} MB)")
    elif os.path.isdir(path):
        print(f"FOUND DIRECTORY: {path}")
    else:
        print(f"MISSING: {path}")

print("\n=== DISK SPACE ===")
total, used, free = shutil.disk_usage("/content")
print(f"Total: {total / (1024**3):.2f} GB")
print(f"Used: {used / (1024**3):.2f} GB")
print(f"Free: {free / (1024**3):.2f} GB")